In [ ]:
pip install -U langchain langchain-community langchain-core langchain-nvidia-ai-endpoints langchain-huggingface chromadb python-dotenv sentence-transformers pydantic

In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime, UTC
import sys
sys.exit # Real Implementation would not exit here, this is just to prevent accidental execution during testing.
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma

load_dotenv()

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
if not NVIDIA_API_KEY:
    raise ValueError("NVIDIA_API_KEY was not found in .env")

LLM_MODEL = os.getenv("NVIDIA_LLM_MODEL", "nvidia/nemotron-3-ultra-550b-a55b")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
NOTES_DIR = Path(os.getenv("NOTES_DIR", "notes"))
VECTOR_DB_DIR = Path(os.getenv("VECTOR_DB_DIR", "chroma_db"))
MEMORY_FILE = Path(os.getenv("MEMORY_FILE", "personal_memory.json"))
CHAT_HISTORY_FILE = Path(os.getenv("CHAT_HISTORY_FILE", "chat_history.json"))


In [ ]:
llm = ChatNVIDIA(
    model=LLM_MODEL,
    api_key=NVIDIA_API_KEY,
    temperature=0.2,
    max_tokens=4096,
)

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],
)


In [ ]:
def load_notes(folder: Path) -> list[Document]:
    documents = []
    folder.mkdir(parents=True, exist_ok=True)
    for path in sorted(folder.glob("*.txt")):
        loader = TextLoader(str(path), encoding="utf-8")
        docs = loader.load()
        for doc in docs:
            doc.metadata.update({"source": path.name, "path": str(path)})
        documents.extend(docs)
    return documents

def split_notes(documents: list[Document]) -> list[Document]:
    chunks = splitter.split_documents(documents)
    for index, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = index
        chunk.metadata["created_at"] = datetime.now(UTC).isoformat()
    return chunks


In [ ]:
documents = load_notes(NOTES_DIR)
chunks = split_notes(documents)
len(documents), len(chunks)

In [ ]:
vectorstore = Chroma(
    collection_name="personal_brain",
    embedding_function=embeddings,
    persist_directory=str(VECTOR_DB_DIR),
)

if chunks:
    existing = vectorstore.get(include=[])
    existing_sources = set()
    for metadata in vectorstore.get(include=["metadatas"]).get("metadatas", []):
        if metadata and metadata.get("source"):
            existing_sources.add(metadata["source"])
    new_chunks = [chunk for chunk in chunks if chunk.metadata.get("source") not in existing_sources]
    if new_chunks:
        vectorstore.add_documents(new_chunks)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6},
)


In [ ]:
class MemoryExtraction(BaseModel):
    goals: list[str] = Field(default_factory=list)
    projects: list[str] = Field(default_factory=list)
    skills: list[str] = Field(default_factory=list)
    interests: list[str] = Field(default_factory=list)
    beliefs: list[str] = Field(default_factory=list)
    observations: list[str] = Field(default_factory=list)
    events: list[str] = Field(default_factory=list)
    tasks: list[str] = Field(default_factory=list)
    relationships: list[dict] = Field(default_factory=list)

memory_llm = llm.with_structured_output(MemoryExtraction)

memory_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Extract durable personal memories from the supplied note. "
        "Do not infer facts that are not stated. Return empty arrays when unsupported."
    ),
    ("human", "{note}")
])

memory_chain = memory_prompt | memory_llm


In [ ]:
def extract_memories(documents: list[Document]) -> list[dict]:
    memories = []
    for document in documents:
        result = memory_chain.invoke({"note": document.page_content})
        item = result.model_dump()
        item["source"] = document.metadata.get("source")
        item["created_at"] = datetime.now(UTC).isoformat()
        memories.append(item)
    return memories

if documents:
    memories = extract_memories(documents)
    MEMORY_FILE.write_text(json.dumps(memories, indent=2, ensure_ascii=False), encoding="utf-8")
else:
    memories = json.loads(MEMORY_FILE.read_text(encoding="utf-8")) if MEMORY_FILE.exists() else []

len(memories)

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a personal-memory assistant. Answer only from the retrieved context. "
        "If the context does not contain the answer, say that the information is not available. "
        "Do not invent personal facts. Cite the source filenames in brackets when useful."
    ),
    (
        "human",
        "Question:\n{question}\n\nRetrieved context:\n{context}"
    )
])

answer_chain = qa_prompt | llm | StrOutputParser()

def format_context(documents: list[Document]) -> str:
    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
        for doc in documents
    )

def answer_question(question: str) -> str:
    retrieved = retriever.invoke(question)
    context = format_context(retrieved)
    answer = answer_chain.invoke({"question": question, "context": context})
    history = json.loads(CHAT_HISTORY_FILE.read_text(encoding="utf-8")) if CHAT_HISTORY_FILE.exists() else []
    history.append({
        "timestamp": datetime.now(UTC).isoformat(),
        "question": question,
        "answer": answer,
        "sources": [doc.metadata.get("source") for doc in retrieved],
    })
    CHAT_HISTORY_FILE.write_text(json.dumps(history, indent=2, ensure_ascii=False), encoding="utf-8")
    return answer


In [ ]:
question = "What are my current goals and projects?"
print(answer_question(question))

In [ ]:
def rebuild_vector_store() -> None:
    global vectorstore, retriever
    if VECTOR_DB_DIR.exists():
        import shutil
        shutil.rmtree(VECTOR_DB_DIR)
    vectorstore = Chroma(
        collection_name="personal_brain",
        embedding_function=embeddings,
        persist_directory=str(VECTOR_DB_DIR),
    )
    documents = load_notes(NOTES_DIR)
    chunks = split_notes(documents)
    if chunks:
        vectorstore.add_documents(chunks)
    retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})

def run_chat() -> None:
    print("Personal Brain RAG. Type exit to stop.")
    while True:
        question = input("> ").strip()
        if question.lower() in {"exit", "quit"}:
            break
        if question:
            print(answer_question(question))
